# Homework 4: Evaluation

Evaluating keyword, vector, and hybrid search on a ground truth dataset.

## Setup

In [ ]:
import sys
import os
import json
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# Add module 2 directory to path for embedder and download
module2_dir = os.path.abspath(os.path.join(os.getcwd(), '..', 'module 2 - vectors'))
sys.path.insert(0, module2_dir)

from download import download
download("Xenova/all-MiniLM-L6-v2", dest=os.path.join(module2_dir, "models"))

from embedder import Embedder
embedder = Embedder(path=os.path.join(module2_dir, "models", "Xenova", "all-MiniLM-L6-v2"))
print("Embedder loaded successfully!")

## Load Course Data (72 pages)

In [ ]:
from gitsource import GithubRepositoryDataReader, chunk_documents

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]
print(f"Number of documents: {len(documents)}")
print(f"First 3 filenames:")
for doc in documents[:3]:
    print(f"  {doc['filename']}")

## Q1. Generating Questions

Generate questions for the first 3 pages using Gemini 3.1 Flash-Lite and report the average number of input tokens.

In [ ]:
from dotenv import load_dotenv
load_dotenv(os.path.join(os.getcwd(), '..', '.env'))

from google import genai
from google.genai import types

gemini_client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.

Return a JSON object with a single key 'questions' containing a list of 5 question strings.
""".strip()

# Generate questions for first 3 pages using Gemini 3.1 Flash-Lite
input_tokens_list = []
all_records = []

for doc in documents[:3]:
    user_prompt = json.dumps({
        "filename": doc["filename"],
        "content": doc["content"]
    })

    response = gemini_client.models.generate_content(
        model='gemini-3.1-flash-lite',
        contents=user_prompt,
        config=types.GenerateContentConfig(
            system_instruction=data_gen_instructions,
            response_mime_type='application/json',
            response_schema={
                'type': 'object',
                'properties': {
                    'questions': {
                        'type': 'array',
                        'items': {'type': 'string'}
                    }
                },
                'required': ['questions']
            }
        )
    )

    input_tokens = response.usage_metadata.prompt_token_count
    input_tokens_list.append(input_tokens)

    questions = json.loads(response.text)['questions']
    print(f"{doc['filename']}: {input_tokens} input tokens, {len(questions)} questions")
    for q in questions:
        all_records.append({"question": q, "filename": doc["filename"]})

avg_input_tokens = sum(input_tokens_list) / len(input_tokens_list)
print(f"\nAverage input tokens: {avg_input_tokens:.0f}")
print(f"\n>>> Answer Q1: ~1400 (closest option)")

## Load Ground Truth (360 questions)

Pre-generated ground truth for all 72 pages.

In [ ]:
df_gt = pd.read_csv("ground-truth.csv")
ground_truth = df_gt.to_dict(orient="records")

print(f"Number of ground truth records: {len(ground_truth)}")
print(f"\nFirst record:")
print(f"  question: {ground_truth[0]['question']}")
print(f"  filename: {ground_truth[0]['filename']}")

## Build Search Indices

Create chunks, embed them, build text index and vector index.

In [ ]:
# Chunk documents
chunks = chunk_documents(documents, size=2000, step=1000)
print(f"Number of chunks: {len(chunks)}")

In [ ]:
# Embed all chunks
BATCH_SIZE = 32
all_embeddings = []

for i in tqdm(range(0, len(chunks), BATCH_SIZE), desc="Embedding chunks"):
    batch = chunks[i:i+BATCH_SIZE]
    texts = [chunk["content"] for chunk in batch]
    batch_embeddings = embedder.encode_batch(texts)
    all_embeddings.append(batch_embeddings)

X = np.vstack(all_embeddings)
print(f"Embedding matrix shape: {X.shape}")

In [ ]:
from minsearch import Index, VectorSearch

# Build text index
text_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"],
)
text_index.fit(chunks)
print("Text index built!")

# Build vector index
vector_index = VectorSearch(
    keyword_fields=["filename"],
)
vector_index.fit(X, chunks)
print("Vector index built!")

In [ ]:
# Define search functions

def text_search(query, num_results=5):
    return text_index.search(query, num_results=num_results)

def vector_search(query, num_results=5):
    query_vector = embedder.encode(query)
    return vector_index.search(query_vector=query_vector, num_results=num_results)

def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

print("Search functions defined!")

## Q2. First result with text search

In [ ]:
q = ground_truth[0]["question"]
print(f"Question: {q}")
print(f"Expected filename: {ground_truth[0]['filename']}")

q2_text_results = text_search(q)

print(f"\nText search top 5 results:")
for i, r in enumerate(q2_text_results):
    print(f"  {i+1}. {r['filename']} (start={r['start']})")

print(f"\n>>> Answer Q2: {q2_text_results[0]['filename']}")

## Q3. First result with vector search

In [ ]:
q3_vector_results = vector_search(q)

print(f"Question: {q}")
print(f"Expected filename: {ground_truth[0]['filename']}")

print(f"\nVector search top 5 results:")
for i, r in enumerate(q3_vector_results):
    print(f"  {i+1}. {r['filename']} (start={r['start']})")

print(f"\n>>> Answer Q3: {q3_vector_results[0]['filename']}")

## Evaluation Functions

Reuse the same evaluation approach from the module, adapted for `filename` matching.

In [ ]:
def compute_relevance(search_func, record):
    """Run search for a question and return a list of 0s and 1s.
    A result is 1 if its filename matches the ground truth filename."""
    question = record["question"]
    expected_filename = record["filename"]

    results = search_func(question)

    relevance = []
    for doc in results:
        if doc["filename"] == expected_filename:
            relevance.append(1)
        else:
            relevance.append(0)

    return relevance


def hit_rate(relevance_lists):
    """Fraction of questions where the correct page appears in the results."""
    hits = sum(1 for rel in relevance_lists if any(r == 1 for r in rel))
    return hits / len(relevance_lists)


def mrr(relevance_lists):
    """Mean Reciprocal Rank - rewards finding the page near the top."""
    total_rr = 0.0
    for rel in relevance_lists:
        for i, r in enumerate(rel):
            if r == 1:
                total_rr += 1 / (i + 1)
                break
    return total_rr / len(relevance_lists)


def evaluate(search_func, ground_truth_data):
    """Evaluate a search function over the whole ground truth."""
    relevance_lists = []
    for record in tqdm(ground_truth_data, desc="Evaluating"):
        rel = compute_relevance(search_func, record)
        relevance_lists.append(rel)

    return {
        "hit_rate": hit_rate(relevance_lists),
        "mrr": mrr(relevance_lists)
    }

print("Evaluation functions defined!")

## Q4. Evaluating text search

What's the Hit Rate?

In [ ]:
text_eval = evaluate(text_search, ground_truth)
print(f"\nText Search Results:")
print(f"  Hit Rate: {text_eval['hit_rate']:.4f}")
print(f"  MRR:      {text_eval['mrr']:.4f}")
print(f"\n>>> Answer Q4: {text_eval['hit_rate']:.2f}")

## Q5. Evaluating vector search

What's the MRR?

In [ ]:
vector_eval = evaluate(vector_search, ground_truth)
print(f"\nVector Search Results:")
print(f"  Hit Rate: {vector_eval['hit_rate']:.4f}")
print(f"  MRR:      {vector_eval['mrr']:.4f}")
print(f"\n>>> Answer Q5: {vector_eval['mrr']:.2f}")

## Q6. Tuning hybrid search

Evaluate `hybrid_search` for k values 1, 50, 100, and 200. Which k gives the best MRR?

In [ ]:
k_values = [1, 50, 100, 200]
hybrid_results = {}

for k_val in k_values:
    print(f"\nEvaluating hybrid search with k={k_val}...")

    def make_hybrid(k_param):
        def _search(query, k=k_param):
            tr = text_search(query, num_results=10)
            vr = vector_search(query, num_results=10)
            return rrf([tr, vr], k=k)
        return _search

    result = evaluate(make_hybrid(k_val), ground_truth)
    hybrid_results[k_val] = result
    print(f"  k={k_val}: Hit Rate={result['hit_rate']:.4f}, MRR={result['mrr']:.4f}")

print("\n" + "="*50)
print("Hybrid Search Results Summary:")
print("="*50)
print(f"{'k':>5}  {'Hit Rate':>10}  {'MRR':>10}")
print("-"*30)
for k_val in k_values:
    r = hybrid_results[k_val]
    print(f"{k_val:>5}  {r['hit_rate']:>10.4f}  {r['mrr']:>10.4f}")

# Find the best k (smallest k if tied)
best_k = min(k_values, key=lambda k: (-hybrid_results[k]['mrr'], k))
print(f"\n>>> Answer Q6: k={best_k} (best MRR = {hybrid_results[best_k]['mrr']:.4f})")

## Summary of All Answers

In [ ]:
print("="*60)
print("HOMEWORK 4 - EVALUATION - ANSWERS SUMMARY")
print("="*60)
print()
print(f"Q1 - Average input tokens: ~1400")
print(f"Q2 - First text search result: {q2_text_results[0]['filename']}")
print(f"Q3 - First vector search result: {q3_vector_results[0]['filename']}")
print(f"Q4 - Text search Hit Rate: {text_eval['hit_rate']:.4f}")
print(f"Q5 - Vector search MRR: {vector_eval['mrr']:.4f}")
print(f"Q6 - Best k for hybrid search: {best_k}")